# LaTeX string demo

Demonstration of the `latex_symbol` functionality on `Gate`, `FidelityIndex`, and `Path`.

In [1]:
from IPython.display import Math, display
from qiskit.circuit.library import CZGate
from qiskit.quantum_info import Clifford, QubitSparsePauli

from qiskit_noise_learning.gate_sets import ModelGate, ModelGateSet
from qiskit_noise_learning.models import CompleteFidelityModel, PauliLindbladModel
from qiskit_noise_learning.sequences import FidelityIndex, Path

## 1. `Gate`

`Gate` now has an optional `latex_symbol` 

In [4]:
gate_set = ModelGateSet(2)

# prep
gate_set.add_gate(ModelGate("P", [], qubit_idxs=[0, 1], prep_idxs=[0, 1]))

# meas
gate_set.add_gate(ModelGate("M", [], qubit_idxs=[0, 1], meas_idxs=[0, 1]))

# CZ gate
gate_set.add_gate(
    ModelGate("CZ", [((0, 1), Clifford(CZGate()))], qubit_idxs=[0, 1], latex_str=r"\mathrm{CZ}")
)

print(f"Gate 'CZ' latex_str: {gate_set['CZ'].latex_str}")
print(f"Gate 'P' latex_str (default): {gate_set['P'].latex_str}")

Gate 'CZ' latex_str: \mathrm{CZ}
Gate 'P' latex_str (default): P


## 2. Fidelity Index Labels

In [26]:
model = PauliLindbladModel.k_local(gate_set, k=2)

fi_x0 = FidelityIndex.from_gate(
    gate=gate_set["CZ"],
    pauli=QubitSparsePauli("IX"),
    in_bit_indices=frozenset(),
    out_bit_indices=frozenset(),
)

fi_z0 = FidelityIndex.from_gate(
    gate=gate_set["CZ"],
    pauli=QubitSparsePauli("IZ"),
    in_bit_indices=frozenset(),
    out_bit_indices=frozenset(),
)

fi_z1 = FidelityIndex.from_gate(
    gate=gate_set["CZ"],
    pauli=QubitSparsePauli("ZI"),
    in_bit_indices=frozenset(),
    out_bit_indices=frozenset(),
)

fi_prep = FidelityIndex.from_gate(
    gate=gate_set["P"],
    pauli=QubitSparsePauli("II"),
    in_bit_indices=frozenset(),
    out_bit_indices=frozenset({0}),
)

fi_meas = FidelityIndex.from_gate(
    gate=gate_set["M"],
    pauli=QubitSparsePauli("II"),
    in_bit_indices=frozenset({0}),
    out_bit_indices=frozenset(),
)

The `"transition"` format is used for building strings when wanting to think about tracking Paulis through gates.

In [27]:
# Transition format
display(Math(model.fidelity_index_latex_str(fi_x0, format="transition")))
display(Math(model.fidelity_index_latex_str(fi_z1, format="transition")))
display(Math(model.fidelity_index_latex_str(fi_prep, format="transition")))
display(Math(model.fidelity_index_latex_str(fi_meas, format="transition")))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

The `"formula"` format produces a string representing the fidelity value itself.

In [28]:
# Formula format
display(Math(model.fidelity_index_latex_str(fi_x0, format="formula")))
display(Math(model.fidelity_index_latex_str(fi_z1, format="formula")))
display(Math(model.fidelity_index_latex_str(fi_meas, format="formula")))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### 2.1 General `FidelityModel`

The above formula representation is for `PauliLindbladModel`, in which the noise model is a Pauli channel (and hence each fidelity is described by a single Pauli operator). For general fidelity models, the format is more verbose.

In [29]:
complete_model = CompleteFidelityModel(gate_set)

display(Math(complete_model.fidelity_index_latex_str(fi_x0, format="formula")))
display(Math(complete_model.fidelity_index_latex_str(fi_z1, format="formula")))
display(Math(complete_model.fidelity_index_latex_str(fi_prep, format="formula")))
display(Math(complete_model.fidelity_index_latex_str(fi_meas, format="formula")))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In this case it displays the full index data, consisting of:
- A Pauli
- The "in bit indices" corresponding to qubit indices on which Z operators being computed in post-processing (or something like this)
- The "out bit indices" corresponding to the qubit indices on which are Z operators being prepared in any reset operators (or something like this)

This can probably use some work - the bits themselves are extremely verbose.

## 3. Paths

In [30]:
fi_y0z1 = FidelityIndex.from_gate(
    gate=gate_set["CZ"],
    pauli=QubitSparsePauli("ZY"),
    in_bit_indices=frozenset(),
    out_bit_indices=frozenset(),
)

fi_meas_zz = FidelityIndex.from_gate(
    gate=gate_set["M"],
    pauli=QubitSparsePauli("II"),
    in_bit_indices=frozenset({0, 1}),
    out_bit_indices=frozenset(),
)

path = Path(
    start_fragment=[fi_prep],
    repeatable_fragment=[fi_x0, fi_y0z1],
    end_fragment=[fi_meas_zz],
    depth=5,
)
display(Math(model.path_latex_str(path, format="transition")))

<IPython.core.display.Math object>

In [31]:
display(Math(model.path_latex_str(path.without_depth(), format="transition")))

<IPython.core.display.Math object>

Notes:
- That the end of the repeatable fragment loops back into itself is not visually represented.
- `r` is being used for "repetitions" in anticipation of clarifying depth v.s. repetitions in the package.

Also: if the single q transition required between indices is identity, it will not be displayed.

In [33]:
path = Path(
    start_fragment=[fi_prep],
    repeatable_fragment=[fi_z0, fi_z0],
    end_fragment=[fi_meas],
    depth=5,
)
display(Math(model.path_latex_str(path, format="transition")))

<IPython.core.display.Math object>

The formula is analogous.

In [34]:
display(Math(model.path_latex_str(path, format="formula")))

<IPython.core.display.Math object>

## Fidelity Index Labels

Create fidelity indices and display them in both formats.

In [3]:
model = CompleteFidelityModel(gate_set)

fi_x0 = FidelityIndex.from_gate(
    gate=gate_set["CZ"],
    pauli=QubitSparsePauli.from_sparse_label(("X", [0]), num_qubits=2),
    in_bit_indices=frozenset(),
    out_bit_indices=frozenset(),
)

fi_z1 = FidelityIndex.from_gate(
    gate=gate_set["CZ"],
    pauli=QubitSparsePauli.from_sparse_label(("Z", [1]), num_qubits=2),
    in_bit_indices=frozenset(),
    out_bit_indices=frozenset(),
)

fi_prep = FidelityIndex.from_gate(
    gate=gate_set["P"],
    pauli=QubitSparsePauli.identity(num_qubits=2),
    in_bit_indices=frozenset(),
    out_bit_indices=frozenset({0}),
)

fi_meas = FidelityIndex.from_gate(
    gate=gate_set["M"],
    pauli=QubitSparsePauli.identity(num_qubits=2),
    in_bit_indices=frozenset({0}),
    out_bit_indices=frozenset(),
)

# Transition format
display(Math(model.fidelity_index_latex_str(fi_x0, format="transition")))
display(Math(model.fidelity_index_latex_str(fi_z1, format="transition")))
display(Math(model.fidelity_index_latex_str(fi_prep, format="transition")))
display(Math(model.fidelity_index_latex_str(fi_meas, format="transition")))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [4]:
fi_x0.transition

(<QubitSparsePauli on 2 qubits: X_0>, <QubitSparsePauli on 2 qubits: X_0>)

In [5]:
# Formula format
display(Math(model.fidelity_index_latex_str(fi_x0, format="formula")))
display(Math(model.fidelity_index_latex_str(fi_z1, format="formula")))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## Path Labels

Create paths and display their LaTeX representations.

In [6]:
# Simple path with just a repeatable fragment
path = Path(
    start_fragment=[fi_prep],
    repeatable_fragment=[fi_x0, fi_z1],
    end_fragment=[fi_meas],
    depth=5,
)
display(Math(model.path_latex_str(path)))

<IPython.core.display.Math object>

In [7]:
# Unbound path (depth=None shows 'r')
path_unbound = Path(
    start_fragment=[fi_prep],
    repeatable_fragment=[fi_x0],
    end_fragment=[fi_meas],
    depth=None,
)
display(Math(model.path_latex_str(path_unbound)))

<IPython.core.display.Math object>

In [8]:
# Path with start and end fragments
path_full = Path(
    start_fragment=[fi_prep],
    repeatable_fragment=[fi_x0, fi_z1],
    end_fragment=[fi_meas],
    depth=3,
)
display(Math(model.path_latex_str(path_full)))

<IPython.core.display.Math object>

In [9]:
# Path with formula format
display(Math(model.path_latex_str(path_full, format="formula")))

<IPython.core.display.Math object>

In [10]:
from qiskit_noise_learning.models import PauliLindbladModel

model = PauliLindbladModel.k_local(gate_set, k=2)

# Transition format
display(Math(model.fidelity_index_latex_str(fi_x0, format="transition")))
display(Math(model.fidelity_index_latex_str(fi_z1, format="transition")))
display(Math(model.fidelity_index_latex_str(fi_prep, format="transition")))
display(Math(model.fidelity_index_latex_str(fi_meas, format="transition")))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [35]:
# Path with formula format
display(Math(model.path_latex_str(path, format="formula")))

<IPython.core.display.Math object>

Notes:
- Could combine fidelity indices that are equal into single exponent.